<a href="https://colab.research.google.com/github/WeegorMartins/customer-decisioning-lab/blob/main/notebooks/01_generate_card_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

```markdown
# 01 — Experimento sintético de cartões

Objetivo: criar clientes artificiais, sortear ações e simular resultados.

Importante: nenhuma linha representa uma pessoa real.
```

# Parte B — importar e configurar

In [1]:
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print("Bibliotecas carregadas.")
print("SEED =", SEED)

Bibliotecas carregadas.
SEED = 42


In [2]:
FAST_MODE = True

if FAST_MODE:
    N_CUSTOMERS = 80_000
else:
    N_CUSTOMERS = 300_000

print("Modo rápido:", FAST_MODE)
print("Quantidade:", N_CUSTOMERS)

Modo rápido: True
Quantidade: 80000


In [3]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [4]:
PROJECT_DIR = Path(
    "/content/drive/MyDrive/customer-decisioning-lab"
)
RAW_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Projeto:", PROJECT_DIR)
print("Raw existe:", RAW_DIR.exists())
print("Processed existe:", PROCESSED_DIR.exists())

Projeto: /content/drive/MyDrive/customer-decisioning-lab
Raw existe: True
Processed existe: True


# Parte C — criar características anteriores à decisão

In [5]:
rng = np.random.default_rng(SEED)

decision_months = pd.date_range(
    start="2025-01-01",
    end="2026-06-01",
    freq="MS"
)

print("Primeiro mês:", decision_months.min())
print("Último mês:", decision_months.max())
print("Número de meses:", len(decision_months))

Primeiro mês: 2025-01-01 00:00:00
Último mês: 2026-06-01 00:00:00
Número de meses: 18


In [6]:
df = pd.DataFrame({
    "customer_id": np.arange(1, N_CUSTOMERS + 1),
    "decision_date": rng.choice(
        decision_months,
        size=N_CUSTOMERS
    ),
    "tenure_months": rng.integers(
        low=1,
        high=121,
        size=N_CUSTOMERS
    ),
    "age": np.clip(
        rng.normal(
            loc=41,
            scale=12,
            size=N_CUSTOMERS
        ).round(),
        18,
        80
    ).astype(int)
})

display(df.head())
print("Formato:", df.shape)

,customer_id,decision_date,tenure_months,age
0,1,2025-02-01,44,45
1,2,2026-02-01,35,40
2,3,2025-12-01,2,43
3,4,2025-08-01,15,37
4,5,2025-08-01,23,31


Formato: (80000, 4)


In [7]:
df["spend_90d"] = rng.gamma(
    shape=2.2,
    scale=420,
    size=N_CUSTOMERS
).round(2)

df["transactions_90d"] = rng.poisson(
    lam=12,
    size=N_CUSTOMERS
)

df["recency_days"] = np.clip(
    rng.exponential(
        scale=24,
        size=N_CUSTOMERS
    ),
    0,
    180
).round().astype(int)

df["digital_sessions_30d"] = rng.poisson(
    lam=5,
    size=N_CUSTOMERS
)

df["merchant_diversity_90d"] = rng.integers(
    low=1,
    high=16,
    size=N_CUSTOMERS
)

display(df[[
    "spend_90d",
    "transactions_90d",
    "recency_days",
    "digital_sessions_30d",
    "merchant_diversity_90d"
]].describe().round(2))

,spend_90d,transactions_90d,recency_days,digital_sessions_30d,merchant_diversity_90d
count,80000.00,80000.00,80000.00,80000.00,80000.00
mean,925.26,11.98,24.04,5.01,8.01
std,622.55,3.45,24.01,2.24,4.31
min,0.53,1.00,0.00,0.00,1.00
25%,466.86,10.00,7.00,3.00,4.00
50%,793.18,12.00,17.00,5.00,8.00
75%,1238.22,14.00,33.00,6.00,12.00
max,6250.98,31.00,180.00,16.00,15.00


In [8]:
df["contacts_30d"] = rng.poisson(
    lam=1.4,
    size=N_CUSTOMERS
)

df["complaints_180d"] = rng.binomial(
    n=2,
    p=0.04,
    size=N_CUSTOMERS
)

df["marketing_consent"] = rng.binomial(
    n=1,
    p=0.88,
    size=N_CUSTOMERS
)

display(
    df[[
        "contacts_30d",
        "complaints_180d",
        "marketing_consent"
    ]].mean().round(3)
)

,0
contacts_30d,1.399
complaints_180d,0.081
marketing_consent,0.883


# Parte D — criar segmentos

In [9]:
conditions = [
    df["tenure_months"].le(3),
    (
        df["recency_days"].le(30)
        & df["spend_90d"].ge(700)
    ),
    df["recency_days"].between(31, 90),
    df["recency_days"].gt(90)
]

segment_names = [
    "novo",
    "ativo",
    "em_risco",
    "inativo"
]

df["lifecycle_segment"] = np.select(
    conditions,
    segment_names,
    default="ocasional"
)

segment_table = (
    df["lifecycle_segment"]
    .value_counts(dropna=False)
    .rename_axis("segment")
    .reset_index(name="customers")
)

segment_table["share"] = (
    segment_table["customers"] / len(df)
)

display(segment_table)

,segment,customers,share
0,ativo,31914,0.398925
1,ocasional,24157,0.301963
2,em_risco,20172,0.252150
3,novo,1936,0.024200
4,inativo,1821,0.022763


# Parte E — sortear ações

In [10]:
ACTION_NAMES = {
    0: "no_contact",
    1: "cashback",
    2: "double_points",
    3: "installment"
}

ACTION_PROBABILITIES = {
    0: 0.25,
    1: 0.25,
    2: 0.25,
    3: 0.25
}

print(ACTION_NAMES)
print(
    "Soma das probabilidades:",
    sum(ACTION_PROBABILITIES.values())
)

{0: 'no_contact', 1: 'cashback', 2: 'double_points', 3: 'installment'}
Soma das probabilidades: 1.0


In [11]:
action_codes = list(ACTION_PROBABILITIES.keys())
action_probs = list(ACTION_PROBABILITIES.values())

df["treatment"] = rng.choice(
    action_codes,
    size=N_CUSTOMERS,
    p=action_probs
)

df["treatment_probability"] = df[
    "treatment"
].map(ACTION_PROBABILITIES)

df["treatment_name"] = df[
    "treatment"
].map(ACTION_NAMES)

randomization_check = (
    df.groupby(
        ["treatment", "treatment_name"],
        as_index=False
    )
    .agg(customers=("customer_id", "size"))
)

randomization_check["share"] = (
    randomization_check["customers"] / len(df)
)

display(randomization_check)

,treatment,treatment_name,customers,share
0,0,no_contact,20145,0.251812
1,1,cashback,19955,0.249438
2,2,double_points,19856,0.248200
3,3,installment,20044,0.250550


# Parte F — criar a resposta natural

In [12]:
base_logit = (
    -2.70
    + 0.00055 * np.minimum(
        df["spend_90d"],
        3000
    )
    + 0.045 * df["digital_sessions_30d"]
    - 0.012 * df["recency_days"]
    - 0.160 * df["contacts_30d"]
)

base_probability = 1 / (
    1 + np.exp(-base_logit)
)

display(
    pd.Series(
        base_probability,
        name="base_probability"
    ).describe().round(4)
)

,base_probability
count,80000.0000
mean,0.0844
std,0.0389
min,0.0055
25%,0.0588
50%,0.0772
75%,0.1011
max,0.3550


# Parte G — criar efeitos diferentes

In [13]:
effect_cashback = (
    0.55
    + 0.30 * (
        df["lifecycle_segment"] == "ativo"
    )
    + 0.22 * (
        df["merchant_diversity_90d"] >= 8
    )
    - 0.38 * (
        df["contacts_30d"] >= 3
    )
)

In [14]:
effect_points = (
    0.30
    + 0.42 * (
        df["lifecycle_segment"] == "ativo"
    )
    + 0.28 * (
        df["digital_sessions_30d"] >= 6
    )
    - 0.18 * (
        df["recency_days"] > 60
    )
)

In [15]:
effect_installment = (
    0.18
    + 0.62 * (
        df["lifecycle_segment"] == "em_risco"
    )
    + 0.22 * (
        df["spend_90d"].between(300, 900)
    )
    - 0.32 * (
        df["complaints_180d"] > 0
    )
)

In [16]:
treatment_effect = np.select(
    [
        df["treatment"].eq(1),
        df["treatment"].eq(2),
        df["treatment"].eq(3)
    ],
    [
        effect_cashback,
        effect_points,
        effect_installment
    ],
    default=0.0
)

response_probability = 1 / (
    1 + np.exp(
        -(base_logit + treatment_effect)
    )
)

response_probability = np.clip(
    response_probability,
    0.001,
    0.95
)

df["converted_90d"] = rng.binomial(
    n=1,
    p=response_probability
)

In [17]:
conversion_by_action = (
    df.groupby(
        ["treatment", "treatment_name"],
        as_index=False
    )
    .agg(
        customers=("customer_id", "size"),
        conversion_rate=(
            "converted_90d",
            "mean"
        )
    )
)

display(
    conversion_by_action.style.format({
        "conversion_rate": "{:.2%}"
    })
)

,treatment,treatment_name,customers,conversion_rate
0,0,no_contact,20145,8.14%
1,1,cashback,19955,16.85%
2,2,double_points,19856,15.07%
3,3,installment,20044,11.89%


# Parte H — valor e guardrails

In [18]:
base_future_spend = (
    0.45 * df["spend_90d"]
    + 18 * df["transactions_90d"]
    + rng.normal(
        loc=0,
        scale=140,
        size=N_CUSTOMERS
    )
)

action_spend_bonus = (
    df["converted_90d"]
    * np.select(
        [
            df["treatment"].eq(1),
            df["treatment"].eq(2),
            df["treatment"].eq(3)
        ],
        [360.0, 260.0, 300.0],
        default=0.0
    )
)

df["future_spend_90d"] = np.maximum(
    0,
    base_future_spend + action_spend_bonus
).round(2)

df["gross_margin_90d"] = (
    0.018 * df["future_spend_90d"]
).round(2)

In [19]:
optout_logit = (
    -5.20
    + 0.60 * df["contacts_30d"]
    + 0.75 * (
        df["complaints_180d"] > 0
    )
    + 0.45 * (
        df["treatment"] != 0
    )
)

optout_probability = 1 / (
    1 + np.exp(-optout_logit)
)

df["optout_90d"] = rng.binomial(
    n=1,
    p=np.clip(
        optout_probability,
        0.001,
        0.50
    )
)

In [20]:
complaint_logit = (
    -6.00
    + 0.90 * (
        df["complaints_180d"] > 0
    )
    + 0.25 * (
        df["treatment"] != 0
    )
    + 0.35 * (
        df["contacts_30d"] >= 3
    )
)

complaint_probability = 1 / (
    1 + np.exp(-complaint_logit)
)

df["complaint_90d"] = rng.binomial(
    n=1,
    p=np.clip(
        complaint_probability,
        0.0001,
        0.30
    )
)

In [21]:
guardrails = (
    df.groupby(
        ["treatment", "treatment_name"],
        as_index=False
    )
    .agg(
        optout_rate=("optout_90d", "mean"),
        complaint_rate=(
            "complaint_90d",
            "mean"
        )
    )
)

display(
    guardrails.style.format({
        "optout_rate": "{:.2%}",
        "complaint_rate": "{:.2%}"
    })
)

,treatment,treatment_name,optout_rate,complaint_rate
0,0,no_contact,1.75%,0.28%
1,1,cashback,2.93%,0.43%
2,2,double_points,2.74%,0.46%
3,3,installment,2.56%,0.38%


# Parte I — validar

In [22]:
def validate_card_data(data: pd.DataFrame) -> dict:
    checks = {
        "customer_id_not_null":
            data["customer_id"].notna().all(),
        "customer_id_unique":
            data["customer_id"].is_unique,
        "valid_treatment":
            data["treatment"].isin([0, 1, 2, 3]).all(),
        "valid_probability":
            data["treatment_probability"].between(0, 1).all(),
        "valid_conversion":
            data["converted_90d"].isin([0, 1]).all(),
        "valid_consent":
            data["marketing_consent"].isin([0, 1]).all(),
        "non_negative_spend":
            data["spend_90d"].ge(0).all(),
        "decision_date_not_null":
            data["decision_date"].notna().all()
    }
    return checks

In [23]:
quality_checks = validate_card_data(df)

quality_table = pd.DataFrame({
    "check": quality_checks.keys(),
    "passed": quality_checks.values()
})

display(quality_table)

assert quality_table["passed"].all(), (
    "Existe pelo menos um teste com falha."
)

print("TODOS OS TESTES PASSARAM")

,check,passed
0,customer_id_not_null,True
1,customer_id_unique,True
2,valid_treatment,True
3,valid_probability,True
4,valid_conversion,True
5,valid_consent,True
6,non_negative_spend,True
7,decision_date_not_null,True


TODOS OS TESTES PASSARAM


In [24]:
balance_table = (
    df.groupby("treatment")
    .agg(
        customers=("customer_id", "size"),
        spend_mean=("spend_90d", "mean"),
        recency_mean=("recency_days", "mean"),
        sessions_mean=(
            "digital_sessions_30d",
            "mean"
        ),
        consent_rate=(
            "marketing_consent",
            "mean"
        )
    )
    .round(3)
)

display(balance_table)

,customers,spend_mean,recency_mean,sessions_mean,consent_rate
treatment,,,,,
0,20145,916.973,24.120,5.023,0.883
1,19955,926.774,23.873,5.012,0.887
2,19856,924.030,24.195,5.008,0.879
3,20044,933.283,23.983,4.989,0.881


In [25]:
PRE_TREATMENT_FEATURES = [
    "tenure_months",
    "age",
    "spend_90d",
    "transactions_90d",
    "recency_days",
    "digital_sessions_30d",
    "merchant_diversity_90d",
    "contacts_30d",
    "complaints_180d",
    "marketing_consent",
    "lifecycle_segment"
]

POST_TREATMENT_COLUMNS = [
    "converted_90d",
    "future_spend_90d",
    "gross_margin_90d",
    "optout_90d",
    "complaint_90d"
]

leakage = set(
    PRE_TREATMENT_FEATURES
).intersection(POST_TREATMENT_COLUMNS)

assert not leakage, (
    f"Vazamento encontrado: {leakage}"
)

print("NENHUM VAZAMENTO NA LISTA DE FEATURES")

NENHUM VAZAMENTO NA LISTA DE FEATURES


# Parte J — salvar

In [26]:
df = df.sort_values(
    ["decision_date", "customer_id"]
).reset_index(drop=True)

display(df.head())
display(df.tail())

,customer_id,decision_date,tenure_months,age,spend_90d,transactions_90d,recency_days,digital_sessions_30d,merchant_diversity_90d,contacts_30d,...,marketing_consent,lifecycle_segment,treatment,treatment_probability,treatment_name,converted_90d,future_spend_90d,gross_margin_90d,optout_90d,complaint_90d
0,56,2025-01-01,30,42,897.83,14,8,3,4,2,...,1,ativo,1,0.25,cashback,0,603.67,10.87,0,0
1,104,2025-01-01,43,45,1490.55,12,4,10,3,3,...,0,ativo,1,0.25,cashback,0,995.82,17.92,1,0
2,117,2025-01-01,65,55,523.72,13,18,6,2,0,...,1,ocasional,2,0.25,double_points,0,501.37,9.02,0,0
3,138,2025-01-01,15,61,1099.80,8,9,5,6,2,...,1,ativo,0,0.25,no_contact,1,741.74,13.35,0,0
4,147,2025-01-01,100,45,182.31,11,0,10,2,1,...,1,ocasional,3,0.25,installment,0,291.66,5.25,0,0


,customer_id,decision_date,tenure_months,age,spend_90d,transactions_90d,recency_days,digital_sessions_30d,merchant_diversity_90d,contacts_30d,...,marketing_consent,lifecycle_segment,treatment,treatment_probability,treatment_name,converted_90d,future_spend_90d,gross_margin_90d,optout_90d,complaint_90d
79995,79944,2026-06-01,11,48,796.85,9,5,3,14,1,...,1,ativo,3,0.25,installment,0,635.47,11.44,0,0
79996,79969,2026-06-01,89,32,604.51,16,12,2,7,2,...,1,ocasional,3,0.25,installment,0,319.77,5.76,0,0
79997,79988,2026-06-01,42,36,1298.01,11,5,8,5,2,...,1,ativo,2,0.25,double_points,0,804.11,14.47,0,0
79998,79993,2026-06-01,54,46,1944.25,12,15,6,14,0,...,1,ativo,0,0.25,no_contact,0,1066.67,19.20,0,0
79999,79994,2026-06-01,17,38,568.71,14,11,3,11,1,...,0,ocasional,1,0.25,cashback,0,505.79,9.10,0,0


In [27]:
size_label = (
    "fast"
    if FAST_MODE
    else "full"
)

full_path = (
    PROCESSED_DIR
    / f"card_experiment_{size_label}.parquet"
)

df.to_parquet(
    full_path,
    index=False
)

print("Salvo em:", full_path)
print(
    "Tamanho MB:",
    round(
        full_path.stat().st_size / 1_000_000,
        2
    )
)

Salvo em: /content/drive/MyDrive/customer-decisioning-lab/data/processed/card_experiment_fast.parquet
Tamanho MB: 2.06


In [28]:
sample_path = (
    PROCESSED_DIR
    / "customer_sample_5000.csv"
)

df.sample(
    n=min(5_000, len(df)),
    random_state=SEED
).to_csv(
    sample_path,
    index=False
)

print("Amostra:", sample_path)

Amostra: /content/drive/MyDrive/customer-decisioning-lab/data/processed/customer_sample_5000.csv


In [29]:
run_summary = {
    "seed": SEED,
    "fast_mode": FAST_MODE,
    "rows": len(df),
    "columns": len(df.columns),
    "start_date": str(
        df["decision_date"].min().date()
    ),
    "end_date": str(
        df["decision_date"].max().date()
    ),
    "conversion_rate": float(
        df["converted_90d"].mean()
    ),
    "optout_rate": float(
        df["optout_90d"].mean()
    )
}

display(run_summary)

{'seed': 42,
 'fast_mode': True,
 'rows': 80000,
 'columns': 21,
 'start_date': '2025-01-01',
 'end_date': '2026-06-01',
 'conversion_rate': 0.1297125,
 'optout_rate': 0.0249375}